# LLM-Powered Applications & Distributed Computing

## Part 1: Distributed Data Processing with Spark


In [1]:
from urllib.request import urlretrieve
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

files = [
    ("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", raw_dir/"yellow_taxi_data.parquet"),
    # ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", raw_dir/"taxi_zone_lookup.csv"),
]

for url, filename in files:
    urlretrieve(url, filename)

print("Done!")

Done!


### Spark Environment Setup & Data Loading

In [1]:
# Creating a SparkSession
import os
from pyspark.sql import SparkSession

# Recreate Spark session with Windows-safe Hadoop local FS settings
try:
    spark.stop()
except Exception:
    pass

os.environ.pop("HADOOP_HOME", None)
os.environ.pop("hadoop.home.dir", None)

spark = SparkSession.builder\
    .appName("NYC Yellow Taxi Trip Records (January 2024)") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.file.impl", "org.apache.hadoop.fs.local.LocalFs") \
    .getOrCreate()

In [2]:
# Verify the session
print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print(f'Master: {spark.sparkContext.master}')
print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')

Spark version: 3.5.1
App name: NYC Yellow Taxi Trip Records (January 2024)
Master: local[*]
Default parallelism: 20


In [3]:
sc = spark.sparkContext  # Access SparkContext
print(sc.getConf().getAll())  # Print all configurations

[('spark.driver.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'), ('spark.app.name', 'NYC Yellow Taxi Trip Records (January 2024)'), ('spark.sql.adaptive.enabled', 'true')

In [4]:
# Loading the NYC Yellow Taxi Parquet data into a Spark DataFrame
file_path = "data\\raw\\yellow_taxi_data.parquet"
df = spark.read.format("parquet").load(file_path)
df.show(5)
df.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [5]:
print(f"Total rows: {df.count()}")
print(f"Total partitions: {df.rdd.getNumPartitions()}")

Total rows: 2964624
Total partitions: 12


In [6]:
import time
import pandas as pd

# Time Spark read (lazy - just metadata)
start = time.time()
df_spark = spark.read.parquet('data/raw/yellow_taxi_data.parquet')
spark_read_time = time.time() - start

# Time Spark action (forces full read)
start = time.time()
spark_count = df_spark.count()

spark_action_time = time.time() - start

# Time Pandas read
start = time.time()
df_pandas = pd.read_parquet('data/raw/yellow_taxi_data.parquet')
pandas_read_time = time.time() - start
print(f'Spark schema read: {spark_read_time:.3f}s (lazy - no data loaded)')
print(f'Spark count action: {spark_action_time:.3f}s ({spark_count:,} rows)')
print(f'Pandas full read: {pandas_read_time:.3f}s ({len(df_pandas):,} rows)')
print(f'Pandas memory usage: {df_pandas.memory_usage(deep=True).sum() / 1e6:.1f}MB')

# Clean up Pandas DataFrame to free memory
del df_pandas

Spark schema read: 0.101s (lazy - no data loaded)
Spark count action: 0.165s (2,964,624 rows)
Pandas full read: 0.267s (2,964,624 rows)
Pandas memory usage: 418.0MB


Interpretation: Spark usually showed a lower load time than Pandas since it uses lazy evaluation unlike Pandas. So, in other words when Spark's load function is called it builds the plan rather than read the data immediately (only if an action is performed), whereas Pandas reads the entire files into memory right away.

### Data Cleaning & Feature Engineering in Spark

In [7]:
from pyspark.sql import functions as F

trips = df.select(
    F.col('tpep_pickup_datetime').alias('pickup_time'),
    F.col('tpep_dropoff_datetime').alias('dropoff_time'),
    'passenger_count',
    'trip_distance',
    'fare_amount',
    'tip_amount',
    'total_amount',
    'payment_type',
    'PULocationID',
    'DOLocationID'
)

print(f'Original rows: {df.count():,}')

Original rows: 2,964,624


In [8]:
df_nulls = trips.filter(
    F.col("pickup_time").isNull() |
    F.col("dropoff_time").isNull() |
    F.col("PULocationID").isNull() |
    F.col("DOLocationID").isNull() |
    F.col("fare_amount").isNull() |
    F.col("trip_distance").isNull()
)
df_nulls.show()

+-----------+------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+
|pickup_time|dropoff_time|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|PULocationID|DOLocationID|
+-----------+------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+
+-----------+------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+



In [9]:
# Remove rows with nulls in critical columns using Spark DataFrame API
critical_cols = [
    'pickup_time', 
    'dropoff_time', 
    'PULocationID', 
    'DOLocationID', 
    'fare_amount', 
    'trip_distance'
 ]

trips_clean = trips.dropna(subset=critical_cols)
print(f"Rows after cleaning: {trips_clean.count()}")
print(f"Number of rows removed: {trips.count() - trips_clean.count()}")

Rows after cleaning: 2964624
Number of rows removed: 0


In [10]:
# Filter out invalid trips using Spark DataFrame API
trips_valid = trips_clean.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") >= 0) &
    (F.col("fare_amount") <= 500) &
    (F.col("dropoff_time") >= F.col("pickup_time"))
 )
print(f"Rows after filtering invalid trips: {trips_valid.count()}")
print(f"Number of rows removed: {trips_clean.count() - trips_valid.count()}")

Rows after filtering invalid trips: 2870102
Number of rows removed: 94522


In [11]:
trips_enriched = trips_valid.withColumns({
    'trip_duration_min': F.round(
        (F.unix_timestamp('dropoff_time') - F.unix_timestamp('pickup_time')) / 60, 2
    ),
    'trip_speed_mph': F.when(
        (F.unix_timestamp('dropoff_time') - F.unix_timestamp('pickup_time')) / 3600 > 0,
        F.round(F.col('trip_distance') / ((F.unix_timestamp('dropoff_time') - F.unix_timestamp('pickup_time')) / 3600), 2)
    ).otherwise(0),
    'pickup_hour': F.hour('pickup_time'),
    'pickup_day': F.dayofweek('pickup_time'),
    'tip_percentage': F.when(
        F.col('fare_amount') != 0,
        F.round((F.col('tip_amount') / F.col('fare_amount')) * 100, 2)
    ).otherwise(0)
})

print(f'Final enriched rows: {trips_enriched.count():,}')
print(f'\nTotal rows removed: {trips.count() - trips_enriched.count():,}')

trips_enriched.show(5)

Final enriched rows: 2,870,102

Total rows removed: 94,522
+-------------------+-------------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+-----------------+--------------+-----------+----------+--------------+
|        pickup_time|       dropoff_time|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|PULocationID|DOLocationID|trip_duration_min|trip_speed_mph|pickup_hour|pickup_day|tip_percentage|
+-------------------+-------------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+-----------------+--------------+-----------+----------+--------------+
|2024-01-01 00:57:55|2024-01-01 01:17:43|              1|         1.72|       17.7|       0.0|        22.7|           2|         186|          79|             19.8|          5.21|          0|         2|           0.0|
|2024-01-01 00:03:00|2024-01-01 00:09:36|              1|          1.

### Spark SQL Analytics

In [12]:
# Register your cleaned DataFrame as a temporary SQL view

# Register as a temporary view
trips_enriched.createOrReplaceTempView('trips')

# Query 1: What are the top 10 busiest pickup hours, and what is the average fare and tip percentage for each?
busiest_locations = spark.sql('''
    SELECT pickup_hour,
    COUNT(*) as num_trips,
    ROUND(AVG(fare_amount), 2) as avg_fare,
    ROUND(AVG(tip_percentage), 2) as avg_tip_percentage
    FROM trips
    GROUP BY pickup_hour
    ORDER BY num_trips DESC
    LIMIT 10
''')

print('Top 10 Busiest Pickup Hours:')
busiest_locations.show()

Top 10 Busiest Pickup Hours:
+-----------+---------+--------+------------------+
|pickup_hour|num_trips|avg_fare|avg_tip_percentage|
+-----------+---------+--------+------------------+
|         18|   206284|   17.01|             22.78|
|         17|   200315|   18.12|             22.34|
|         16|   184971|   19.46|             21.83|
|         15|   184009|   19.11|              19.8|
|         19|   178812|   17.63|             22.86|
|         14|   178031|   19.27|              19.8|
|         13|   165361|   18.42|             19.78|
|         12|   159916|    17.8|             19.74|
|         21|   155915|   18.29|             21.88|
|         20|   155561|   18.05|             22.17|
+-----------+---------+--------+------------------+



Interpretation: The top 10 busiest pickup hours were mostly hours in the evening when person may be commuting from work or engaging in nightlife activities. However, some hours where in the afternoon (12-14) indicating daytime demand as well which may likely be due to lunch-time travel/errands.

In [30]:
# Show physical execution plan for Query 1
print("Physical Execution Plan for Query 1 (Top 10 Busiest Pickup Hours):")
print("=" * 80)
busiest_locations.explain(mode="formatted")

Physical Execution Plan for Query 1 (Top 10 Busiest Pickup Hours):
== Physical Plan ==
AdaptiveSparkPlan (11)
+- TakeOrderedAndProject (10)
   +- HashAggregate (9)
      +- Exchange (8)
         +- HashAggregate (7)
            +- InMemoryTableScan (1)
                  +- InMemoryRelation (2)
                        +- * Project (6)
                           +- * Filter (5)
                              +- * ColumnarToRow (4)
                                 +- Scan parquet  (3)


(1) InMemoryTableScan
Output [3]: [fare_amount#10, pickup_hour#381, tip_percentage#383]
Arguments: [fare_amount#10, pickup_hour#381, tip_percentage#383]

(2) InMemoryRelation
Arguments: [pickup_time#202, dropoff_time#203, passenger_count#3L, trip_distance#4, fare_amount#10, tip_amount#13, total_amount#16, payment_type#9L, PULocationID#7, DOLocationID#8, trip_duration_min#379, trip_speed_mph#380, pickup_hour#381, pickup_day#382, tip_percentage#383], CachedRDDBuilder(org.apache.spark.sql.execution.columnar.De

In [13]:
# Query 2: Which day of the week has the highest average trip speed? Include average distance and duration.
highest_speed = spark.sql(''' 
    SELECT pickup_day,
    ROUND(AVG(trip_speed_mph), 2) as avg_speed_mph,
    ROUND(AVG(trip_distance), 2) as avg_distance,
    ROUND(AVG(trip_duration_min), 2) as avg_duration_minutes
    FROM trips
    GROUP BY pickup_day
    ORDER BY avg_speed_mph DESC
    LIMIT 1
''')

print('Day of the Week with Highest Average Trip Speed:')
highest_speed.show()

Day of the Week with Highest Average Trip Speed:
+----------+-------------+------------+--------------------+
|pickup_day|avg_speed_mph|avg_distance|avg_duration_minutes|
+----------+-------------+------------+--------------------+
|         3|        17.46|        4.25|               16.17|
+----------+-------------+------------+--------------------+



Interpretation: The highest average speed trip was 17.46 mph on a Tuesday, which lasted for 16.17 minutes. This may be due to better travel condition i.e less traffic compared to the other days in NYC.

In [14]:
# Query 3: Using a window function, rank the top 5 pickup locations by total revenue for each day of the week.
from pyspark.sql.window import Window

# Aggregate revenue by day and pickup location
daily_location_revenue = trips_enriched \
    .groupBy("pickup_day", "PULocationID") \
    .agg(F.sum("total_amount").alias("total_revenue"))

# Define window for ranking locations within each day
location_window = Window.partitionBy('pickup_day').orderBy(F.desc('total_revenue'))

# Rank locations and calculate percentage of that day's max revenue
ranked_trips = daily_location_revenue \
    .withColumn('revenue_rank', F.row_number().over(location_window)) \
    .withColumn('day_max_revenue', F.max('total_revenue').over(Window.partitionBy('pickup_day'))) \
    .withColumn('pct_of_day_max', F.round(F.col('total_revenue') / F.col('day_max_revenue') * 100, 2))

# Show top 5 pickup locations for each day
ranked_trips \
    .filter(F.col('revenue_rank') <= 5) \
    .select('pickup_day', 'PULocationID', 'total_revenue', 'revenue_rank', 'pct_of_day_max') \
    .orderBy('pickup_day', 'revenue_rank') \
    .show(35)

+----------+------------+------------------+------------+--------------+
|pickup_day|PULocationID|     total_revenue|revenue_rank|pct_of_day_max|
+----------+------------+------------------+------------+--------------+
|         1|         132|1564287.9299999834|           1|         100.0|
|         1|         138| 763398.5400000012|           2|          48.8|
|         1|         230| 346553.9500000004|           3|         22.15|
|         1|         186|         264131.38|           4|         16.89|
|         1|          79|263467.74000000034|           5|         16.84|
|         2|         132|2054606.7299999362|           1|         100.0|
|         2|         138|1021138.2800000039|           2|          49.7|
|         2|         161| 460145.2800000035|           3|          22.4|
|         2|         236| 373008.8900000014|           4|         18.15|
|         2|         237|  372575.480000002|           5|         18.13|
|         3|         132|1795093.5599999563|       

Interpretation: For all days of the week the PULocationID 132 was noted to be the top revenue generating pickup zone i.e it was always ranked no. 1 and is set as the daily baseline. PULocation 138 was consistently ranked no. 2 as well, where it usually generated about 39-61% of the top zone's revenue, while the other locations contributed smaller amounts. 

In [15]:
# Query 4: Calculate the cumulative trip count by hour of day (running total from hour 0 to 23). At what hour does the cumulative count surpass 50% of daily trips?

# Aggregate to hourly level first
hourly_trip = trips_enriched.groupBy('pickup_hour').agg(
    F.count('*').alias('hourly_trips')
).orderBy('pickup_hour')

# Window for cumulative sum (all rows up to current)
cumulative_window = Window.orderBy('pickup_hour').rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

#calculate cumulative count
hourly_trip = hourly_trip.withColumn(
    'cumulative_trips',
    F.sum('hourly_trips').over(cumulative_window)
)

# Calculate total trips to find 50% threshold
total_trips = hourly_trip.agg(F.sum('hourly_trips')).collect()[0][0]
threshold = total_trips * 0.5

# Find the hour where cumulative trips surpass 50% of total
first_hour = hourly_trip.filter(F.col('cumulative_trips') >= threshold) \
    .orderBy('pickup_hour') \
    .select('pickup_hour') \
    .first()[0]

print(f"Cumulative trips surpass 50% at hour {first_hour}")
hourly_trip.show(24)

Cumulative trips surpass 50% at hour 15
+-----------+------------+----------------+
|pickup_hour|hourly_trips|cumulative_trips|
+-----------+------------+----------------+
|          0|       75251|           75251|
|          1|       50491|          125742|
|          2|       34976|          160718|
|          3|       22948|          183666|
|          4|       15285|          198951|
|          5|       17496|          216447|
|          6|       39415|          255862|
|          7|       80872|          336734|
|          8|      113508|          450242|
|          9|      125621|          575863|
|         10|      135425|          711288|
|         11|      146754|          858042|
|         12|      159916|         1017958|
|         13|      165361|         1183319|
|         14|      178031|         1361350|
|         15|      184009|         1545359|
|         16|      184971|         1730330|
|         17|      200315|         1930645|
|         18|      206284|         2

Interpretation: The cumulative distribution shows that 50% of all trips is reached by hour 15 (3 PM), meaning half of daily taxi demand occurs from midnight through mid‑afternoon. Trip volume is lowest overnight (about 2–5 AM), rises steadily in the morning, and peaks in the late afternoon/evening (around 5–6 PM), indicating demand is concentrated in daytime and commuter hours.

In [16]:
# Query 5: Compare average fare, distance, and tip percentage between short trips (<2
# miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has the highest tip percentage?

trip_stats = spark.sql('''
    SELECT 
    CASE 
        WHEN trip_distance < 2 THEN 'Short (<2 miles)'
        WHEN trip_distance BETWEEN 2 AND 10 THEN 'Medium (2-10 miles)'
        ELSE 'Long (>10 miles)'
    END AS trip_category,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
    FROM trips
    GROUP BY trip_category
    ORDER BY avg_tip_percentage DESC
''')

print('Trip Statistics by category comparing fare, distance, and tip percentage:')
trip_stats.show()

Trip Statistics by category comparing fare, distance, and tip percentage:
+-------------------+----------+--------+------------+------------------+
|      trip_category|trip_count|avg_fare|avg_distance|avg_tip_percentage|
+-------------------+----------+--------+------------+------------------+
|   Short (<2 miles)|   1642473|    9.91|        1.13|             23.07|
|   Long (>10 miles)|    225080|   64.65|        21.7|             21.93|
|Medium (2-10 miles)|   1002549|   22.18|        3.96|             18.57|
+-------------------+----------+--------+------------+------------------+



Interpretation: Short trips (< 2miles) has the highest average tip percentage (21.93%) while long (> 10 miles) and medium (2-10 miles) trips have an average tip percentage of 21.93% and 18.57% respectively. This indicates that passengers tip drivers large amounts for very short trips despite the average fare being much smaller. This could be due to passengers being able to reach their location quickly, thus satisfied. Long trips maintain a strong average tip percentage, which makes sense given the average distance and fare.

### Performance Optimization

In [17]:
import time
from pyspark.sql import functions as F

# multi query for benchmarking
def multi_query(df):
    return (
        df.filter(F.col("pickup_hour") == 17)
          .groupBy("PULocationID")
          .agg(
              F.count("*").alias("trip_count"),
              F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
              F.round(F.avg("tip_percentage"), 2).alias("avg_tip_percentage")
          )
          .count()
    )

# Warmup run to eliminate JVM class loading overhead
trips_enriched.count()

# run queries without caching 
no_cache_times = []
for i in range(3):
    start = time.time()
    multi_query(trips_enriched)
    no_cache_times.append(time.time() - start)

no_cache_time_avg = sum(no_cache_times) / len(no_cache_times)
print(f"Average time before caching: {no_cache_time_avg:.3f} seconds")

# cache the DataFrame
trips_enriched.cache()

# materialize cache
start = time.time()
trips_enriched.count()
cache_first_time = time.time() - start
print(f"First run (materializing cache): {cache_first_time:.3f} seconds")

# Subsequent runs use the cache (should be noticeably faster)
cache_times = []
for i in range(3):
    start = time.time()
    multi_query(trips_enriched)
    cache_times.append(time.time() - start)

after_cache_avg = sum(cache_times) / len(cache_times)
print(f"Average time after caching: {after_cache_avg:.3f} seconds")

speedup = no_cache_time_avg / after_cache_avg if after_cache_avg > 0 else None
print(f"Speedup from caching: {speedup:.2f}x")

Average time before caching: 0.447 seconds
First run (materializing cache): 3.676 seconds
Average time after caching: 0.212 seconds
Speedup from caching: 2.11x


Interpretation: Caching provided a small performance gain. Runtime improved from 0.240s to 0.219s after caching, giving a 1.10x speedup (~10% faster). This indicates caching helps for repeated queries, but the benefit is modest for this workload.

In [ ]:
# # Write the cleaned data to Parquet format partitioned by pickup_hour
# import os

# output_dir = "output/trips_enriched_partitioned"

# trips_enriched.write \
#     .mode("overwrite") \
#     .partitionBy("pickup_hour") \
#     .parquet(output_dir)

# print("Partitioned Parquet written successfully.\n")

# # inspect output folders
# for item in sorted(os.listdir(output_dir)):
#     if item.startswith("pickup_hour="):
#         files = os.listdir(os.path.join(output_dir, item))
#         parquet_files = [f for f in files if f.endswith(".parquet")]
#         print(f"{item}: {len(parquet_files)} parquet file(s)")

Py4JJavaError: An error occurred while calling o464.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:286)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:699)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.j

In [21]:
# Alternative: write partitioned parquet without Spark writer (Windows-safe)
import shutil
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
from pyspark.sql import functions as F

output_dir = Path("output/trips_enriched_partitioned").resolve()
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Get partition keys
pickup_hours = [
    row["pickup_hour"]
    for row in trips_enriched.select("pickup_hour").distinct().collect()
]

# Write one parquet file per pickup_hour partition
for hour in sorted(pickup_hours):
    part_dir = output_dir / f"pickup_hour={hour}"
    part_dir.mkdir(parents=True, exist_ok=True)

    pdf = trips_enriched.filter(F.col("pickup_hour") == hour).toPandas()
    table = pa.Table.from_pandas(pdf, preserve_index=False)
    pq.write_table(table, part_dir / "part-00000.parquet", compression="snappy")

print("Partitioned Parquet written successfully with PyArrow.\n")

# Inspect output folders
for item in sorted(output_dir.iterdir()):
    if item.is_dir() and item.name.startswith("pickup_hour="):
        parquet_files = list(item.glob("*.parquet"))
        print(f"{item.name}: {len(parquet_files)} parquet file(s)")

Partitioned Parquet written successfully with PyArrow.

pickup_hour=0: 1 parquet file(s)
pickup_hour=1: 1 parquet file(s)
pickup_hour=10: 1 parquet file(s)
pickup_hour=11: 1 parquet file(s)
pickup_hour=12: 1 parquet file(s)
pickup_hour=13: 1 parquet file(s)
pickup_hour=14: 1 parquet file(s)
pickup_hour=15: 1 parquet file(s)
pickup_hour=16: 1 parquet file(s)
pickup_hour=17: 1 parquet file(s)
pickup_hour=18: 1 parquet file(s)
pickup_hour=19: 1 parquet file(s)
pickup_hour=2: 1 parquet file(s)
pickup_hour=20: 1 parquet file(s)
pickup_hour=21: 1 parquet file(s)
pickup_hour=22: 1 parquet file(s)
pickup_hour=23: 1 parquet file(s)
pickup_hour=3: 1 parquet file(s)
pickup_hour=4: 1 parquet file(s)
pickup_hour=5: 1 parquet file(s)
pickup_hour=6: 1 parquet file(s)
pickup_hour=7: 1 parquet file(s)
pickup_hour=8: 1 parquet file(s)
pickup_hour=9: 1 parquet file(s)


In [27]:
#read back a single partition to verify that partition pruning reduces data scanned
df_partition = spark.read.parquet(str("output/trips_enriched_partitioned/pickup_hour=17/part-00000.parquet"))
print(f"Rows in pickup_hour=17 partition: {df_partition.count()}")

#query that benefits from partition pruning
evening_trips = df_partition.filter(F.col("pickup_hour") == 17)

#Check the execution plan to confirm partition pruning is happening
print("Execution plan for evening_trips query:")
evening_trips.explain(mode="formatted")

#benchmark: partitioned read vs full read
start = time.time()
evening_trips.count()
partitioned_time = time.time() - start

start = time.time()
trips_enriched.filter(F.col("pickup_hour") == 17).count()
full_time = time.time() - start

print(f"Time to read partitioned data: {partitioned_time:.3f} seconds")
print(f"Time to read full data with filter: {full_time:.3f} seconds")
print(f"Speedup from partition pruning: {full_time / partitioned_time:.2f}x" if partitioned_time > 0 else "N/A")


Py4JJavaError: An error occurred while calling o700.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 163.0 failed 1 times, most recent failure: Lost task 0.0 in stage 163.0 (TID 716) (STACANLAB08.sastudents.uwi.tt executor driver): org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1826)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:206)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:283)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:224)
	at org.apache.spark.sql.execution.da

In [29]:
import time
import pyarrow.dataset as ds
import pyarrow.compute as pc

root = "output/trips_enriched_partitioned"

#read partitioned dataset using hive-style partitioning
dataset = ds.dataset(root, format="parquet", partitioning="hive")

print("Dataset schema:")
print(dataset.schema)

#partition-pruned read only pickup_hour=17
start = time.time()
filtered = dataset.to_table(filter=pc.field("pickup_hour") == 17)
partitioned_time = time.time() - start

print("\nRows in pickup_hour=17 partition (PyArrow):", filtered.num_rows)

start = time.time()
full = dataset.to_table()
full_filtered = filtered.filter(pc.equal(filtered["pickup_hour"], 17))
full_time = time.time() - start

print(f"\n\nTime to read partitioned data with PyArrow: {partitioned_time:.3f} seconds")
print(f"Time to read full data and filter with PyArrow: {full_time:.3f} seconds")
print(f"Speedup from partition pruning with PyArrow: {full_time / partitioned_time:.2f}x" if partitioned_time > 0 else "N/A")

Dataset schema:
pickup_time: timestamp[ns]
dropoff_time: timestamp[ns]
passenger_count: double
trip_distance: double
fare_amount: double
tip_amount: double
total_amount: double
payment_type: int64
PULocationID: int32
DOLocationID: int32
trip_duration_min: double
trip_speed_mph: double
pickup_hour: int32
pickup_day: int32
tip_percentage: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1976

Rows in pickup_hour=17 partition (PyArrow): 200315


Time to read partitioned data with PyArrow: 0.024 seconds
Time to read full data and filter with PyArrow: 0.114 seconds
Speedup from partition pruning with PyArrow: 4.69x
